In [0]:
customers_df = spark.read.table(
    "ecommerce_catalog.silver.customers"
)

products_df = spark.read.table(
    "ecommerce_catalog.silver.products"
)

orders_df = spark.read.table(
    "ecommerce_catalog.silver.orders"
)


print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Orders:", orders_df.count())

display(products_df)

Customers: 6
Products: 5
Orders: 3


product_id,product_name,category,price,load_timestamp,source_file_name
P001,Laptop,Electronics,50000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P002,Wireless Mouse,Electronics,800.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P003,Keyboard,Electronics,1500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P005,Unknown,Furniture,9000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P006,Headphones,Unknown,2500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv


In [0]:
orders_customers_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

sales_detail = orders_customers_df.join(
    products_df,
    on="product_id",
    how="inner"
)

display(sales_detail)

display(orders_customers_df)

product_id,customer_id,order_id,order_date,quantity,load_timestamp,source_file_name,customer_name,city,state,load_timestamp,source_file_name,product_name,category,price,load_timestamp,source_file_name
P001,C001,O1001,2026-08-01,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Arjun Retail,Hyderabad,Telangana,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv,Laptop,Electronics,50000.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P002,C002,O1002,2026-08-01,3.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Sai Enterprises,Bengaluru,Karnataka,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv,Wireless Mouse,Electronics,800.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv
P003,C003,O1003,2026-08-02,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv,Keyboard,Electronics,1500.0,2026-08-14T10:08:52.019173Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/products_bad.csv


customer_id,order_id,product_id,order_date,quantity,load_timestamp,source_file_name,customer_name,city,state,load_timestamp,source_file_name
C001,O1001,P001,2026-08-01,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Arjun Retail,Hyderabad,Telangana,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C002,O1002,P002,2026-08-01,3.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Sai Enterprises,Bengaluru,Karnataka,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv
C003,O1003,P003,2026-08-02,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,Lakshmi Stores,Chennai,Tamil Nadu,2026-08-14T10:03:21.727564Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/customers_bad.csv


In [0]:
from pyspark.sql.functions import col

sales_detail = sales_detail.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

sales_detail = sales_detail.select(
    "order_id",
    "order_date",
    "customer_id",
    "customer_name",
    "product_name",
    "category",
    "quantity",
    "sales_amount"
)

display(sales_detail)

order_id,order_date,customer_id,customer_name,product_name,category,quantity,sales_amount
O1003,2026-08-02,C003,Lakshmi Stores,Keyboard,Electronics,2.0,3000.0
O1001,2026-08-01,C001,Arjun Retail,Laptop,Electronics,1.0,50000.0
O1002,2026-08-01,C002,Sai Enterprises,Wireless Mouse,Electronics,3.0,2400.0


In [0]:
sales_detail.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_catalog.gold.sales_detail")

In [0]:
%sql
SELECT *
FROM ecommerce_catalog.gold.sales_detail;

order_id,order_date,customer_id,customer_name,product_name,category,quantity,sales_amount
O1003,2026-08-02,C003,Lakshmi Stores,Keyboard,Electronics,2.0,3000.0
O1001,2026-08-01,C001,Arjun Retail,Laptop,Electronics,1.0,50000.0
O1002,2026-08-01,C002,Sai Enterprises,Wireless Mouse,Electronics,3.0,2400.0


In [0]:
from pyspark.sql.functions import sum, count

daily_sales_summary = (
    sales_detail
    .groupBy("order_date")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    )
    .orderBy("order_date")
)

display(daily_sales_summary)

order_date,total_orders,total_quantity,total_sales
2026-08-01,2,4.0,52400.0
2026-08-02,1,2.0,3000.0


In [0]:
daily_sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_catalog.gold.daily_sales_summary"
    )

In [0]:
%sql
select * from ecommerce_catalog.gold.daily_sales_summary

order_date,total_orders,total_quantity,total_sales
2026-08-01,2,4.0,52400.0
2026-08-02,1,2.0,3000.0


In [0]:
print("Gold sales_detail rows:", sales_detail.count())

from pyspark.sql.functions import col

sales_detail.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Gold sales_detail rows: 3
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
sales_detail.filter(
    col("order_id").isNull() |
    col("order_date").isNull() |
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("product_name").isNull() |
    col("category").isNull() |
    col("quantity").isNull() |
    col("sales_amount").isNull()
).show()

+--------+----------+-----------+-------------+------------+--------+--------+------------+
|order_id|order_date|customer_id|customer_name|product_name|category|quantity|sales_amount|
+--------+----------+-----------+-------------+------------+--------+--------+------------+
+--------+----------+-----------+-------------+------------+--------+--------+------------+



In [0]:
from pyspark.sql.functions import col

silver_orders = spark.read.table(
    "ecommerce_catalog.silver.orders"
)

silver_products = spark.read.table(
    "ecommerce_catalog.silver.products"
)

sales_validation = (
    silver_orders
    .join(
        silver_products.select("product_id", "price"),
        on="product_id",
        how="inner"
    )
    .withColumn(
        "expected_sales_amount",
        col("quantity") * col("price")
    )
)

display(sales_validation)

product_id,order_id,customer_id,order_date,quantity,load_timestamp,source_file_name,price,expected_sales_amount
P003,O1003,C003,2026-08-02,2.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,1500.0,3000.0
P001,O1001,C001,2026-08-01,1.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,50000.0,50000.0
P002,O1002,C002,2026-08-01,3.0,2026-08-14T10:10:14.689856Z,abfss://flmproject@duvvaadlsgen2.dfs.core.windows.net/ecommerce_source/orders_bad.csv,800.0,2400.0


In [0]:
gold_sales = spark.read.table(
    "ecommerce_catalog.gold.sales_detail"
)

comparison = (
    gold_sales
    .join(
        sales_validation.select(
            "order_id",
            "expected_sales_amount"
        ),
        on="order_id",
        how="inner"
    )
)

display(comparison)

order_id,order_date,customer_id,customer_name,product_name,category,quantity,sales_amount,expected_sales_amount
O1003,2026-08-02,C003,Lakshmi Stores,Keyboard,Electronics,2.0,3000.0,3000.0
O1001,2026-08-01,C001,Arjun Retail,Laptop,Electronics,1.0,50000.0,50000.0
O1002,2026-08-01,C002,Sai Enterprises,Wireless Mouse,Electronics,3.0,2400.0,2400.0


In [0]:
comparison.filter(
    col("sales_amount") != col("expected_sales_amount")
).show()

+--------+----------+-----------+-------------+------------+--------+--------+------------+---------------------+
|order_id|order_date|customer_id|customer_name|product_name|category|quantity|sales_amount|expected_sales_amount|
+--------+----------+-----------+-------------+------------+--------+--------+------------+---------------------+
+--------+----------+-----------+-------------+------------+--------+--------+------------+---------------------+



In [0]:
silver_customers = spark.read.table(
    "ecommerce_catalog.silver.customers"
)

invalid_customers = silver_orders.join(
    silver_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Orders with invalid customers:", invalid_customers.count())

Orders with invalid customers: 0
